# detectar_pastillasv2

Detecta colores de pastillas en tiempo real y dibuja recuadros como en `Detectar_colores`, usando las mascaras HSV de `detector_pastillas`.

Presiona `q` para salir.

In [75]:
import json
from pathlib import Path


def load_latest_camera_calibration(defaults):
    base_reports = Path('reports')
    if not base_reports.exists():
        return defaults.copy(), None

    candidates = sorted(
        base_reports.rglob('camera_calibration.json'),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if not candidates:
        return defaults.copy(), None

    latest = candidates[0]
    try:
        rows = json.loads(latest.read_text(encoding='utf-8'))
        if not rows:
            return defaults.copy(), str(latest)

        # Elegimos el sample con mejor score para tomar settings
        best = max(rows, key=lambda row: float(row.get('best_score', 0.0)))
        loaded = defaults.copy()
        loaded['WHITE_L_MIN'] = int(best.get('white_l_min', defaults['WHITE_L_MIN']))
        loaded['WHITE_SAT_MAX'] = int(best.get('white_sat_max', defaults['WHITE_SAT_MAX']))
        d_raw = str(best.get('distance_thresholds', ''))
        d_list = []
        for token in d_raw.split(','):
            token = token.strip()
            if not token:
                continue
            try:
                d_list.append(float(token))
            except ValueError:
                pass
        loaded['DISTANCE_THRESHOLDS'] = d_list if d_list else list(defaults['DISTANCE_THRESHOLDS'])
        loaded['MIN_FILL_RATIO'] = float(best.get('min_fill_ratio', defaults['MIN_FILL_RATIO']))
        loaded['MIN_SOLIDITY'] = float(best.get('min_solidity', defaults['MIN_SOLIDITY']))
        return loaded, str(latest)
    except Exception:
        return defaults.copy(), str(latest)


def build_color_mask(image_hsv, hsv_ranges):
    mask = np.zeros(image_hsv.shape[:2], dtype=np.uint8)
    for lower, upper in hsv_ranges:
        lower_np = np.array(lower, dtype=np.uint8)
        upper_np = np.array(upper, dtype=np.uint8)
        mask = cv2.bitwise_or(mask, cv2.inRange(image_hsv, lower_np, upper_np))
    return mask


def find_print_region(hsv_work, lab_work, white_l_min, white_sat_max):
    # Basado en image_findings._find_print_region
    l_channel = lab_work[:, :, 0]
    sat = hsv_work[:, :, 1]

    white_mask = ((l_channel >= white_l_min) & (sat <= white_sat_max)).astype(np.uint8) * 255
    content_mask = cv2.bitwise_not(white_mask)
    content_mask = cv2.morphologyEx(content_mask, cv2.MORPH_CLOSE, KERNEL_BIG, iterations=2)
    content_mask = cv2.morphologyEx(content_mask, cv2.MORPH_OPEN, KERNEL, iterations=1)

    contours, _ = cv2.findContours(content_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = hsv_work.shape[:2]
        full = np.full((h, w), 255, dtype=np.uint8)
        return (0, 0, w, h), full

    contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(contour)

    region_mask = np.zeros(content_mask.shape, dtype=np.uint8)
    cv2.drawContours(region_mask, [contour], -1, 255, -1)
    region_mask = cv2.dilate(region_mask, KERNEL_BIG, iterations=1)
    return (x, y, w, h), region_mask


def build_background_gate(hsv_work, lab_work, distance_thresholds):
    h, w = hsv_work.shape[:2]
    border = max(8, min(h, w) // 14)

    border_hsv = np.concatenate([
        hsv_work[:border, :, :].reshape(-1, 3),
        hsv_work[-border:, :, :].reshape(-1, 3),
        hsv_work[:, :border, :].reshape(-1, 3),
        hsv_work[:, -border:, :].reshape(-1, 3)
    ], axis=0).astype(np.float32)

    border_lab = np.concatenate([
        lab_work[:border, :, :].reshape(-1, 3),
        lab_work[-border:, :, :].reshape(-1, 3),
        lab_work[:, :border, :].reshape(-1, 3),
        lab_work[:, -border:, :].reshape(-1, 3)
    ], axis=0).astype(np.float32)

    bg_hsv = np.median(border_hsv, axis=0)
    bg_lab = np.median(border_lab, axis=0)

    sat_delta = np.abs(hsv_work[:, :, 1].astype(np.float32) - bg_hsv[1])
    val_delta = np.abs(hsv_work[:, :, 2].astype(np.float32) - bg_hsv[2])
    lab_dist = np.linalg.norm(lab_work.astype(np.float32) - bg_lab.reshape(1, 1, 3), axis=2)

    gate_dist = np.zeros(hsv_work.shape[:2], dtype=np.uint8)
    for t in distance_thresholds:
        gate_dist = cv2.bitwise_or(gate_dist, (lab_dist > float(t)).astype(np.uint8) * 255)

    gate_sv = ((sat_delta > max(14.0, np.std(border_hsv[:, 1]) * 1.8)) | (val_delta > 24.0)).astype(np.uint8) * 255

    fg_gate = cv2.bitwise_or(gate_dist, gate_sv)
    fg_gate = cv2.morphologyEx(fg_gate, cv2.MORPH_CLOSE, KERNEL_BIG, iterations=1)
    fg_gate = cv2.morphologyEx(fg_gate, cv2.MORPH_OPEN, KERNEL, iterations=1)

    return fg_gate, bg_hsv, bg_lab, lab_dist


def choose_final_color(color_scores, med_h, med_s, med_v):
    if not color_scores:
        return None

    sorted_colors = sorted(color_scores.items(), key=lambda pair: pair[1], reverse=True)
    final = sorted_colors[0][0]

    if final in {'RED', 'ORANGE'}:
        if 8.0 <= med_h <= 24.0 and med_s >= 75.0:
            final = 'ORANGE'
        elif (med_h <= 7.0 or med_h >= 171.0) and med_s >= 95.0:
            final = 'RED'

    if final == 'YELLOW' and med_s < 92.0:
        if 8.0 <= med_h <= 38.0 and 10.0 <= med_s <= 150.0 and med_v >= 130.0:
            final = 'CREAM'

    if final == 'GRAY':
        if 8.0 <= med_h <= 38.0 and 10.0 <= med_s <= 140.0 and med_v >= 130.0:
            final = 'CREAM'

    if final == 'CREAM' and med_s < 8.0:
        final = 'GRAY'

    if final == 'BROWN':
        if med_v > 165.0 and 9.0 <= med_h <= 24.0 and med_s >= 95.0:
            final = 'ORANGE'
        elif med_h <= 7.0 or med_h >= 171.0:
            final = 'RED'

    return final


def make_debug_tile(image, title, size=(360, 220), is_mask=False):
    if is_mask:
        tile = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        tile = image.copy()
    tile = cv2.resize(tile, size, interpolation=cv2.INTER_AREA)
    cv2.rectangle(tile, (0, 0), (size[0] - 1, size[1] - 1), (60, 60, 60), 1)
    cv2.putText(tile, title, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2, cv2.LINE_AA)
    return tile


WORK_AREA_RATIO_W = 0.32
WORK_AREA_RATIO_H = 0.32

DEBUG_VIEW = True
DEBUG_PANEL_WINDOW = 'detectar_pastillasv2 - debug'

defaults = {
    'WHITE_L_MIN': int(globals().get('WHITE_L_MIN', 190)),
    'WHITE_SAT_MAX': int(globals().get('WHITE_SAT_MAX', 45)),
    'DISTANCE_THRESHOLDS': list(globals().get('DISTANCE_THRESHOLDS', [18.0, 22.0, 26.0, 30.0])),
    'MIN_FILL_RATIO': float(globals().get('MIN_FILL_RATIO', 0.45)),
    'MIN_SOLIDITY': float(globals().get('MIN_SOLIDITY', 0.70))
}
calib, calib_path = load_latest_camera_calibration(defaults)
WHITE_L_MIN = calib['WHITE_L_MIN']
WHITE_SAT_MAX = calib['WHITE_SAT_MAX']
DISTANCE_THRESHOLDS = calib['DISTANCE_THRESHOLDS']
MIN_FILL_RATIO = calib['MIN_FILL_RATIO']
MIN_SOLIDITY = calib['MIN_SOLIDITY']
print(f"Calibracion activa: {calib_path or 'defaults'}")
print(f"white_l_min={WHITE_L_MIN} white_sat_max={WHITE_SAT_MAX} distance_thresholds={DISTANCE_THRESHOLDS} min_fill_ratio={MIN_FILL_RATIO} min_solidity={MIN_SOLIDITY}")

cam = None
for cam_index in [0, 1]:
    test_cam = cv2.VideoCapture(cam_index)
    if test_cam.isOpened():
        cam = test_cam
        break
    test_cam.release()

if cam is None:
    raise RuntimeError('No se pudo abrir la camara en los indices 0 ni 1')

while True:
    ok, frame = cam.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)
    suavizado = cv2.GaussianBlur(frame, (5, 5), 0)
    hsv = cv2.cvtColor(suavizado, cv2.COLOR_BGR2HSV)
    salida = frame.copy()

    frame_h, frame_w = frame.shape[:2]
    work_w = int(frame_w * WORK_AREA_RATIO_W)
    work_h = int(frame_h * WORK_AREA_RATIO_H)
    work_x = (frame_w - work_w) // 2
    work_y = (frame_h - work_h) // 2

    hsv_work = hsv[work_y:work_y + work_h, work_x:work_x + work_w]
    bgr_work = suavizado[work_y:work_y + work_h, work_x:work_x + work_w]
    lab_work = cv2.cvtColor(bgr_work, cv2.COLOR_BGR2LAB)
    work_area = max(1, work_w * work_h)

    cv2.rectangle(salida, (work_x, work_y), (work_x + work_w, work_y + work_h), (80, 180, 255), 2)
    cv2.putText(salida, 'AREA DE TRABAJO', (work_x + 8, max(24, work_y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (80, 180, 255), 2, cv2.LINE_AA)
    cv2.putText(salida, 'Tecla: d=debug  q=salir', (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 2, cv2.LINE_AA)

    (px, py, pw, ph), print_region_mask = find_print_region(hsv_work, lab_work, WHITE_L_MIN, WHITE_SAT_MAX)
    cv2.rectangle(salida, (work_x + px, work_y + py), (work_x + px + pw, work_y + py + ph), (255, 200, 80), 2)

    color_masks = {}
    union_chroma = np.zeros(hsv_work.shape[:2], dtype=np.uint8)
    union_neutral = np.zeros(hsv_work.shape[:2], dtype=np.uint8)

    for color_name, hsv_ranges in COLOR_MASKS_HSV.items():
        m = build_color_mask(hsv_work, hsv_ranges)
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, KERNEL, iterations=1)
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, KERNEL, iterations=2)
        color_masks[color_name] = m
        if color_name in CHROMATIC_COLORS:
            union_chroma = cv2.bitwise_or(union_chroma, m)
        else:
            union_neutral = cv2.bitwise_or(union_neutral, m)

    fg_gate, bg_hsv, bg_lab, lab_dist = build_background_gate(hsv_work, lab_work, DISTANCE_THRESHOLDS)
    fg_gate = cv2.bitwise_and(fg_gate, print_region_mask)

    object_mask = cv2.bitwise_or(
        cv2.bitwise_and(union_chroma, cv2.dilate(fg_gate, KERNEL, iterations=1)),
        cv2.bitwise_and(union_neutral, fg_gate)
    )

    object_mask = cv2.bitwise_and(object_mask, print_region_mask)
    object_mask = cv2.morphologyEx(object_mask, cv2.MORPH_CLOSE, KERNEL_BIG, iterations=2)
    object_mask = cv2.morphologyEx(object_mask, cv2.MORPH_OPEN, KERNEL, iterations=1)
    object_mask = cv2.medianBlur(object_mask, 7)

    contours, _ = cv2.findContours(object_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    claimed_mask = np.zeros(hsv_work.shape[:2], dtype=np.uint8)

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_AREA:
            continue
        if area > work_area * MAX_AREA_RATIO:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < MIN_WIDTH or h < MIN_HEIGHT:
            continue

        hull = cv2.convexHull(contour)
        hull_area = max(float(cv2.contourArea(hull)), 1.0)
        solidity = float(area / hull_area)
        fill_ratio = float(area / max(float(w * h), 1.0))
        if fill_ratio < MIN_FILL_RATIO:
            continue
        if solidity < MIN_SOLIDITY:
            continue

        local_mask = np.zeros(hsv_work.shape[:2], dtype=np.uint8)
        cv2.drawContours(local_mask, [contour], -1, 255, -1)

        overlap_claim = cv2.countNonZero(cv2.bitwise_and(local_mask, claimed_mask))
        if overlap_claim > 0.35 * max(cv2.countNonZero(local_mask), 1):
            continue

        pixels_hsv = hsv_work[local_mask > 0]
        if pixels_hsv.size == 0:
            continue

        med_h = float(np.median(pixels_hsv[:, 0]))
        med_s = float(np.median(pixels_hsv[:, 1]))
        med_v = float(np.median(pixels_hsv[:, 2]))

        mean_bg_dist = float(np.mean(lab_dist[local_mask > 0]))
        if mean_bg_dist < DISTANCE_THRESHOLDS[0] and med_s < 38.0:
            continue

        color_scores = {}
        for c in COLOR_PRIORITY:
            overlap = cv2.countNonZero(cv2.bitwise_and(local_mask, color_masks[c]))
            if overlap > 0:
                color_scores[c] = overlap

        final_color = choose_final_color(color_scores, med_h, med_s, med_v)
        if final_color is None:
            continue

        if final_color == 'BLACK' and med_v > 95.0:
            continue
        if final_color == 'GRAY' and (med_s > 34.0 or med_v > 180.0):
            continue

        gx, gy = work_x + x, work_y + y
        draw_color = COLOR_DRAW_BGR.get(final_color, (255, 255, 255))

        cv2.rectangle(salida, (gx, gy), (gx + w, gy + h), draw_color, 2)
        label = final_color
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        top = max(0, gy - th - 8)
        cv2.rectangle(salida, (gx, top), (gx + tw + 8, gy), draw_color, -1)
        text_color = (0, 0, 0) if final_color in {'YELLOW', 'PINK', 'CREAM'} else (255, 255, 255)
        cv2.putText(salida, label, (gx + 4, gy - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 2, cv2.LINE_AA)

        cv2.drawContours(claimed_mask, [contour], -1, 255, -1)

    if DEBUG_VIEW:
        tile_roi = make_debug_tile(bgr_work, 'ROI', is_mask=False)
        tile_fg = make_debug_tile(fg_gate, 'fg_gate', is_mask=True)
        tile_pr = make_debug_tile(print_region_mask, 'print_region_mask', is_mask=True)
        tile_ch = make_debug_tile(union_chroma, 'union cromatica', is_mask=True)
        tile_nt = make_debug_tile(union_neutral, 'union neutral', is_mask=True)
        tile_obj = make_debug_tile(object_mask, 'mascara final', is_mask=True)

        panel_top = np.hstack([tile_roi, tile_fg, tile_pr])
        panel_bottom = np.hstack([tile_ch, tile_nt, tile_obj])
        debug_panel = np.vstack([panel_top, panel_bottom])

        bg_text = f'bg_hsv=({int(bg_hsv[0])},{int(bg_hsv[1])},{int(bg_hsv[2])}) WL={WHITE_L_MIN} WS={WHITE_SAT_MAX}'
        cv2.putText(debug_panel, bg_text, (10, debug_panel.shape[0] - 16), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.imshow(DEBUG_PANEL_WINDOW, debug_panel)

    cv2.imshow('detectar_pastillasv2 - colores', salida)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    if key == ord('d'):
        DEBUG_VIEW = not DEBUG_VIEW
        if not DEBUG_VIEW:
            cv2.destroyWindow(DEBUG_PANEL_WINDOW)

cam.release()
cv2.destroyAllWindows()



Calibracion activa: reports\20260418_183059_camera\camera_calibration.json
white_l_min=200 white_sat_max=40 distance_thresholds=[20.0, 24.0, 28.0, 32.0] min_fill_ratio=0.48 min_solidity=0.72


## Debug HSV por capsula (v2)

Estas celdas te permiten calibrar rangos HSV por capsula usando su imagen de referencia.

Teclas:
- `s`: imprime rango actual para copiar
- `n`: siguiente color de la capsula
- `q`: salir del debug actual


In [76]:
import os
import json
import cv2
import numpy as np

DEBUG_RANGES_FILE = 'hsv_debug_ranges_v2.json'


def _noop(_):
    pass


def _load_debug_ranges(file_path=DEBUG_RANGES_FILE):
    if not os.path.exists(file_path):
        return {}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict):
            return data
        return {}
    except Exception:
        return {}


def _save_debug_ranges(data, file_path=DEBUG_RANGES_FILE):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def _ensure_capsule_slot(db, capsule_name, color_name):
    if capsule_name not in db:
        db[capsule_name] = {}
    if color_name not in db[capsule_name]:
        db[capsule_name][color_name] = {}
    return db[capsule_name][color_name]


def _create_hsv_trackbars(window_name, lower, upper, open_init=1, close_init=2):
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1100, 760)

    cv2.createTrackbar('H_min', window_name, int(lower[0]), 179, _noop)
    cv2.createTrackbar('S_min', window_name, int(lower[1]), 255, _noop)
    cv2.createTrackbar('V_min', window_name, int(lower[2]), 255, _noop)
    cv2.createTrackbar('H_max', window_name, int(upper[0]), 179, _noop)
    cv2.createTrackbar('S_max', window_name, int(upper[1]), 255, _noop)
    cv2.createTrackbar('V_max', window_name, int(upper[2]), 255, _noop)

    cv2.createTrackbar('Open', window_name, int(open_init), 12, _noop)
    cv2.createTrackbar('Close', window_name, int(close_init), 12, _noop)


def _read_hsv_trackbars(window_name):
    h_min = cv2.getTrackbarPos('H_min', window_name)
    s_min = cv2.getTrackbarPos('S_min', window_name)
    v_min = cv2.getTrackbarPos('V_min', window_name)
    h_max = cv2.getTrackbarPos('H_max', window_name)
    s_max = cv2.getTrackbarPos('S_max', window_name)
    v_max = cv2.getTrackbarPos('V_max', window_name)

    if h_min > h_max:
        h_min, h_max = h_max, h_min
    if s_min > s_max:
        s_min, s_max = s_max, s_min
    if v_min > v_max:
        v_min, v_max = v_max, v_min

    k_open = max(0, cv2.getTrackbarPos('Open', window_name))
    k_close = max(0, cv2.getTrackbarPos('Close', window_name))

    return (h_min, s_min, v_min), (h_max, s_max, v_max), k_open, k_close


def _write_slot(db, capsule_name, color_name, lower, upper, k_open, k_close):
    slot = _ensure_capsule_slot(db, capsule_name, color_name)
    slot['lower'] = [int(lower[0]), int(lower[1]), int(lower[2])]
    slot['upper'] = [int(upper[0]), int(upper[1]), int(upper[2])]
    slot['open'] = int(k_open)
    slot['close'] = int(k_close)


def _read_slot_defaults(db, capsule_name, color_name, lower_init, upper_init, open_init=1, close_init=2):
    capsule_data = db.get(capsule_name, {})
    color_data = capsule_data.get(color_name, {})

    lower = tuple(color_data.get('lower', list(lower_init)))
    upper = tuple(color_data.get('upper', list(upper_init)))
    k_open = int(color_data.get('open', open_init))
    k_close = int(color_data.get('close', close_init))

    return lower, upper, k_open, k_close


def _run_single_hsv_debug(image_bgr, capsule_name, color_name, lower_init, upper_init, db, file_path):
    lower0, upper0, open0, close0 = _read_slot_defaults(
        db=db,
        capsule_name=capsule_name,
        color_name=color_name,
        lower_init=lower_init,
        upper_init=upper_init,
        open_init=1,
        close_init=2
    )

    window_name = f'HSV Debug | {capsule_name} | {color_name}'
    _create_hsv_trackbars(window_name, lower0, upper0, open_init=open0, close_init=close0)

    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    while True:
        lower, upper, k_open, k_close = _read_hsv_trackbars(window_name)

        lower_np = np.array(lower, dtype=np.uint8)
        upper_np = np.array(upper, dtype=np.uint8)
        mask = cv2.inRange(hsv, lower_np, upper_np)

        kernel_size_open = 2 * k_open + 1
        kernel_size_close = 2 * k_close + 1
        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size_open, kernel_size_open))
        kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size_close, kernel_size_close))

        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close, iterations=1)

        result = cv2.bitwise_and(image_bgr, image_bgr, mask=mask)
        contour_view = image_bgr.copy()
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(contour_view, contours, -1, (0, 255, 255), 2)

        mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        top = np.hstack([image_bgr, contour_view])
        bottom = np.hstack([mask_bgr, result])
        panel = np.vstack([top, bottom])

        title_main = f'AJUSTA COLOR: {color_name}'
        title_sub = f'Capsula: {capsule_name}'
        info_1 = f'LOW={lower} HIGH={upper}'
        info_2 = f'Open={k_open} Close={k_close} | teclas: s=guardar n=siguiente q=salir'

        cv2.putText(panel, title_main, (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.00, (0, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(panel, title_sub, (10, 64), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(panel, info_1, (10, 96), cv2.FONT_HERSHEY_SIMPLEX, 0.66, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(panel, info_2, (10, 126), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow(window_name, panel)
        key = cv2.waitKey(20) & 0xFF

        if key == ord('s'):
            _write_slot(db, capsule_name, color_name, lower, upper, k_open, k_close)
            _save_debug_ranges(db, file_path)
            print(f"Guardado [{capsule_name}] {color_name}: (({lower[0]}, {lower[1]}, {lower[2]}), ({upper[0]}, {upper[1]}, {upper[2]}))")
        elif key == ord('n'):
            _write_slot(db, capsule_name, color_name, lower, upper, k_open, k_close)
            _save_debug_ranges(db, file_path)
            cv2.destroyWindow(window_name)
            return {
                'color': color_name,
                'lower': lower,
                'upper': upper,
                'open': k_open,
                'close': k_close
            }
        elif key == ord('q'):
            _write_slot(db, capsule_name, color_name, lower, upper, k_open, k_close)
            _save_debug_ranges(db, file_path)
            cv2.destroyWindow(window_name)
            return {
                'color': color_name,
                'lower': lower,
                'upper': upper,
                'open': k_open,
                'close': k_close,
                'aborted': True
            }


def run_capsule_hsv_debug(capsule_name, image_path, color_slots, file_path=DEBUG_RANGES_FILE):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f'No existe la imagen: {image_path}')

    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise RuntimeError(f'No se pudo leer la imagen: {image_path}')

    scale = 560.0 / max(image_bgr.shape[:2])
    if scale < 1.0:
        image_bgr = cv2.resize(
            image_bgr,
            (int(image_bgr.shape[1] * scale), int(image_bgr.shape[0] * scale)),
            interpolation=cv2.INTER_AREA
        )

    db = _load_debug_ranges(file_path)

    print(f'Iniciando debug HSV para {capsule_name}')
    print(f'Imagen: {image_path}')
    print(f'Archivo de salida: {file_path}')

    results = []
    for slot in color_slots:
        result = _run_single_hsv_debug(
            image_bgr=image_bgr,
            capsule_name=capsule_name,
            color_name=slot['name'],
            lower_init=slot['lower'],
            upper_init=slot['upper'],
            db=db,
            file_path=file_path
        )
        results.append(result)
        if result.get('aborted', False):
            break

    _save_debug_ranges(db, file_path)

    print('\nResumen de rangos guardados:')
    for row in results:
        low = row['lower']
        high = row['upper']
        print(f"- {row['color']}: (({low[0]}, {low[1]}, {low[2]}), ({high[0]}, {high[1]}, {high[2]})) | open={row['open']} close={row['close']}")

    print('\nSiguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.')
    cv2.destroyAllWindows()
    return results


In [77]:
run_capsule_hsv_debug(
    capsule_name='blackNyellow_capsule',
    image_path='pills/blackNyellow_capsule.jpg',
    color_slots=[
        {'name': 'BLACK', 'lower': (0, 0, 0), 'upper': (180, 255, 52)},
        {'name': 'YELLOW', 'lower': (22, 60, 110), 'upper': (35, 255, 255)}
    ]
)


Iniciando debug HSV para blackNyellow_capsule
Imagen: pills/blackNyellow_capsule.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- BLACK: ((0, 0, 0), (179, 255, 72)) | open=1 close=1

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.


[{'color': 'BLACK',
  'lower': (0, 0, 0),
  'upper': (179, 255, 72),
  'open': 1,
  'close': 1,
  'aborted': True}]

In [78]:
run_capsule_hsv_debug(
    capsule_name='cream_capsule',
    image_path='pills/cream_capsule.jpg',
    color_slots=[
        {'name': 'GRAY', 'lower': (0, 0, 58), 'upper': (180, 45, 180)},
        {'name': 'CREAM', 'lower': (10, 15, 120), 'upper': (35, 150, 255)}
    ]
)


Iniciando debug HSV para cream_capsule
Imagen: pills/cream_capsule.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- GRAY: ((63, 0, 82), (179, 26, 205)) | open=1 close=3

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.


[{'color': 'GRAY',
  'lower': (63, 0, 82),
  'upper': (179, 26, 205),
  'open': 1,
  'close': 3,
  'aborted': True}]

In [79]:
run_capsule_hsv_debug(
    capsule_name='green_capsule',
    image_path='pills/green_capsule.jpg',
    color_slots=[
        {'name': 'GREEN', 'lower': (35, 45, 60), 'upper': (85, 255, 255)}
    ]
)


Iniciando debug HSV para green_capsule
Imagen: pills/green_capsule.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- GREEN: ((71, 25, 47), (88, 255, 255)) | open=1 close=1

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.


[{'color': 'GREEN',
  'lower': (71, 25, 47),
  'upper': (88, 255, 255),
  'open': 1,
  'close': 1,
  'aborted': True}]

In [80]:
run_capsule_hsv_debug(
    capsule_name='orangeandblue',
    image_path='pills/orangeandblue.jpg',
    color_slots=[
        {'name': 'ORANGE', 'lower': (8, 75, 100), 'upper': (22, 255, 255)},
        {'name': 'BLUE', 'lower': (85, 45, 50), 'upper': (135, 255, 255)}
    ]
)


Iniciando debug HSV para orangeandblue
Imagen: pills/orangeandblue.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- ORANGE: ((0, 70, 158), (25, 251, 255)) | open=2 close=1

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.


[{'color': 'ORANGE',
  'lower': (0, 70, 158),
  'upper': (25, 251, 255),
  'open': 2,
  'close': 1,
  'aborted': True}]

## Debug HSV de todas las referencias v2

Esta celda corre el debug de todas las imagenes v2 una por una.
Usa y actualiza `hsv_debug_ranges_v2.json` en cada ajuste.


In [81]:
V2_ALL_DEBUG_ITEMS = [
    {
        'name': '60429-203_M_LH3',
        'path': 'pills/60429-203_M_LH3.jpg',
        'slots': [
            {'name': 'GREEN', 'lower': (35, 45, 60), 'upper': (85, 255, 255)}
        ]
    },
    {
        'name': '317220267',
        'path': 'pills/317220267.jpg',
        'slots': [
            {'name': 'RED', 'lower': (0, 90, 60), 'upper': (7, 255, 255)}
        ]
    },
    {
        'name': '634810623',
        'path': 'pills/634810623.jpg',
        'slots': [
            {'name': 'BLUE', 'lower': (85, 45, 50), 'upper': (135, 255, 255)}
        ]
    },
    {
        'name': 'blackNyellow_capsule',
        'path': 'pills/blackNyellow_capsule.jpg',
        'slots': [
            {'name': 'BLACK', 'lower': (0, 0, 0), 'upper': (180, 255, 52)},
            {'name': 'YELLOW', 'lower': (22, 60, 110), 'upper': (35, 255, 255)}
        ]
    },
    {
        'name': 'bloe_oval',
        'path': 'pills/bloe_oval.jpg',
        'slots': [
            {'name': 'BLUE', 'lower': (85, 45, 50), 'upper': (135, 255, 255)}
        ]
    },
    {
        'name': 'brown_oval',
        'path': 'pills/brown_oval.jpg',
        'slots': [
            {'name': 'BROWN', 'lower': (10, 70, 35), 'upper': (22, 255, 185)}
        ]
    },
    {
        'name': 'cream_capsule',
        'path': 'pills/cream_capsule.jpg',
        'slots': [
            {'name': 'GRAY', 'lower': (0, 0, 58), 'upper': (180, 45, 180)},
            {'name': 'CREAM', 'lower': (10, 15, 120), 'upper': (35, 150, 255)}
        ]
    },
    {
        'name': 'green_capsule',
        'path': 'pills/green_capsule.jpg',
        'slots': [
            {'name': 'GREEN', 'lower': (35, 45, 60), 'upper': (85, 255, 255)}
        ]
    },
    {
        'name': 'green_oval',
        'path': 'pills/green_oval.jpg',
        'slots': [
            {'name': 'GREEN', 'lower': (35, 45, 60), 'upper': (85, 255, 255)}
        ]
    },
    {
        'name': 'orangeandblue',
        'path': 'pills/orangeandblue.jpg',
        'slots': [
            {'name': 'ORANGE', 'lower': (8, 75, 100), 'upper': (22, 255, 255)},
            {'name': 'BLUE', 'lower': (85, 45, 50), 'upper': (135, 255, 255)}
        ]
    }
]

for item in V2_ALL_DEBUG_ITEMS:
    print('\n' + '=' * 70)
    print(f"DEBUG: {item['name']}")
    run_capsule_hsv_debug(
        capsule_name=item['name'],
        image_path=item['path'],
        color_slots=item['slots'],
        file_path='hsv_debug_ranges_v2.json'
    )



DEBUG: 60429-203_M_LH3
Iniciando debug HSV para 60429-203_M_LH3
Imagen: pills/60429-203_M_LH3.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- GREEN: ((35, 45, 60), (85, 255, 255)) | open=1 close=2

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.

DEBUG: 317220267
Iniciando debug HSV para 317220267
Imagen: pills/317220267.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- RED: ((3, 25, 49), (179, 255, 255)) | open=0 close=4

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.

DEBUG: 634810623
Iniciando debug HSV para 634810623
Imagen: pills/634810623.jpg
Archivo de salida: hsv_debug_ranges_v2.json

Resumen de rangos guardados:
- BLUE: ((85, 45, 50), (135, 255, 255)) | open=1 close=2

Siguiente uso: vuelve a ejecutar la celda y cargara estos valores automaticamente.

DEBUG: blackNyellow_capsule
Iniciando debug HSV para blackNyellow_capsule
Imagen: pills/blackNy